# 06 — ETL Socioeconómico

Genera DOS tablas de contexto:

### `contexto_socioeconomico.csv` (largo, discriminador `indicador`)
| indicador | fuente | territori | desglose |
|-----------|--------|-----------|----------|
| `renta_neta_media` | ine_renta_media_ccaa | Cataluña + España | 4 indicadores de renta |
| `paro_registrat` | paro registrado por sexo | Barcelona (municipi) | sexe |
| `nivell_educatiu` | Poblacio por nivel de estudio | **Barcelona: barri / districte / municipi** | titulació |

La educación está a nivel **barri** (72), districte (9) y municipi (1). Los barris casan con
los barris de GUB en `dim_territorio` (`id_territori`) → **joinable con fact_incidentes_gub**.

### `contexto_poblacion.csv` (para calcular TASAS per cápita)
Padrón municipal (Dataset 1): Cataluña (agregado) + todos los municipios catalanes, por año y sexo (1998-2025).

**Esquemas:**
- socioeconomico: `anyo, id_territori, territori, nivel_territorial, indicador, categoria, sexe, valor, unitat`
- poblacion: `anyo, id_territori, territori, nivel_territorial, sexe, valor`

Se unen al análisis por `anyo + territori` (o `id_territori` → dim_territorio: Cataluña CCAA y barris GUB).

**Alcance:** renta CCAA/estatal, paro Barcelona-municipi, educación Barcelona-barri, población
Cataluña+municipis. Renta municipal fina (Archivo B) y población por edad/nacionalidad: extensión futura.

⚠️ **No mezclar niveles** (`nivel_territorial`) al agregar: barri ⊂ districte ⊂ municipi; municipi ⊂ ccaa.

**Reglas:** P1 miles con punto · decimal coma en paro · '-'/'..' = null · skiprows=7 en paro."

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r'E:\informacion y documentos\Curso analisis de datos IT Academy\Proyecto final\criminalistica_cat')
RAW  = BASE / 'data' / 'raw'
CLEAN = BASE / 'data' / 'clean'
S = RAW / 'socioeconomico'

dim_territorio = pd.read_csv(CLEAN / 'dim_territorio.csv', encoding='utf-8')
ID_CATALUNA = int(dim_territorio[(dim_territorio['fuente']=='ministerio_ine') & (dim_territorio['nivel_territorial']=='ccaa')]['id_territorio'].iloc[0])
print('ID Cataluña CCAA:', ID_CATALUNA)

SEXE_MAP = {'hombres':'Homes','mujeres':'Dones','total':'Total','Homes':'Homes','Dones':'Dones','Total':'Total'}
ESQ_SOCIO = ['anyo','id_territori','territori','nivel_territorial','indicador','categoria','sexe','valor','unitat']
ESQ_POB = ['anyo','id_territori','territori','nivel_territorial','sexe','valor']

---
## 1. Renta → `renta_neta_media` (Cataluña + España, ine_renta_media_ccaa)

In [ ]:
renta = pd.read_csv(S / 'ine_renta_media_ccaa_2008_2025.csv', sep='\t', encoding='utf-8-sig', thousands='.')
col_terr = renta.columns[0]
col_ind = renta.columns[1]
renta = renta[renta[col_terr].isin(['09 Cataluña', 'Total Nacional'])].copy()
renta['territori'] = renta[col_terr].map({'09 Cataluña': 'Cataluña', 'Total Nacional': 'España'})
renta['nivel_territorial'] = renta['territori'].map({'Cataluña': 'ccaa', 'España': 'estat'})
renta['id_territori'] = renta['nivel_territorial'].map({'ccaa': ID_CATALUNA}).astype('Int64')
renta['anyo'] = renta['Periodo'].astype(int)
renta['indicador'] = 'renta_neta_media'
renta['categoria'] = renta[col_ind]
renta['sexe'] = pd.NA
renta['valor'] = pd.to_numeric(renta['Total'], errors='coerce')
renta['unitat'] = 'euros'
socio_renta = renta[ESQ_SOCIO].copy()
print('socio_renta:', len(socio_renta), 'filas | años', socio_renta['anyo'].min(), '-', socio_renta['anyo'].max())
print('  indicadores:', socio_renta['categoria'].nunique(), '| territoris:', sorted(socio_renta['territori'].unique()))
print('  nulos valor:', socio_renta['valor'].isna().sum())
socio_renta.head(3)

---
## 2. Paro → `paro_registrat` (Barcelona municipi, por sexe)

P7: el fichero tiene 7 líneas de metadatos; cabecera real en la línea 7. Decimal coma.

In [ ]:
paro = pd.read_csv(S / 'paro registrado por sexo.csv', sep=';', skiprows=7, encoding='utf-8-sig', decimal=',')
paro = paro.rename(columns={paro.columns[0]: 'anyo'})
print('Columnas paro:', paro.columns.tolist())
# Quedarnos sólo con filas cuyo 'anyo' es numérico (descarta posibles metadatos al final)
paro['anyo'] = pd.to_numeric(paro['anyo'], errors='coerce')
paro = paro.dropna(subset=['anyo'])
paro['anyo'] = paro['anyo'].astype(int)

SEX_COLS = {'Sexe. Homes': 'Homes', 'Sexe. Dones': 'Dones', 'Sexe. Total': 'Total'}
paro_long = paro.melt(id_vars='anyo', value_vars=list(SEX_COLS.keys()), var_name='sexe_raw', value_name='valor')
paro_long['sexe'] = paro_long['sexe_raw'].map(SEX_COLS)
paro_long['valor'] = pd.to_numeric(paro_long['valor'], errors='coerce')
paro_long['territori'] = 'Barcelona'
paro_long['nivel_territorial'] = 'municipi'
paro_long['id_territori'] = pd.NA
paro_long['indicador'] = 'paro_registrat'
paro_long['categoria'] = pd.NA
paro_long['unitat'] = 'persones_mitjana_anual'
socio_paro = paro_long[ESQ_SOCIO].copy()

# Validación: Homes + Dones == Total por año (con tolerancia: medias con 1 decimal)
piv = socio_paro.pivot_table(index='anyo', columns='sexe', values='valor')
max_dif = float((piv['Homes'] + piv['Dones'] - piv['Total']).abs().max())
anyo_min, anyo_max = int(socio_paro['anyo'].min()), int(socio_paro['anyo'].max())
print(f'socio_paro: {len(socio_paro)} filas | años {anyo_min}-{anyo_max}')
print(f'  Homes+Dones vs Total: dif máx = {max_dif:.2f} (rounding de medias, OK)')
socio_paro.head(3)

---
## 3. Educación → `nivell_educatiu` (Barcelona municipi, melt WIDE→LONG)

P6: años como columnas ('01 ene YYYY'). '-' = null, miles con punto.

In [ ]:
edu = pd.read_csv(S / 'Poblacio por nivel de estudio.csv', sep=',', encoding='utf-8-sig')
year_cols = [c for c in edu.columns if 'ene' in c]
print('Columnas año:', year_cols)
edu_long = edu.melt(
    id_vars=['Territorio', 'Tipo de territorio', 'Titulación académica'],
    value_vars=year_cols, var_name='fecha', value_name='valor_raw'
)
edu_long['anyo'] = edu_long['fecha'].str.extract(r'(\d{4})').astype(int)
# '-' = null, miles con punto → parsear
edu_long['valor_raw'] = edu_long['valor_raw'].astype(str).str.strip().replace('-', np.nan)
edu_long['valor'] = pd.to_numeric(edu_long['valor_raw'].str.replace('.', '', regex=False), errors='coerce')
edu_long = edu_long.dropna(subset=['valor'])

# OJO: este fichero NO es sólo Barcelona-municipi: 'Tipo de territorio' distingue
# Barri (72) / Districte (9) / Municipi (1, Barcelona) / '-' (agregados).
NIVEL_MAP = {'Barri': 'barri', 'Districte': 'districte', 'Municipi': 'municipi'}
edu_long['nivel_territorial'] = edu_long['Tipo de territorio'].map(NIVEL_MAP)  # '-' → NaN
edu_long['territori'] = edu_long['Territorio']

# id_territori: los 72 barris casan (71 exactos) con los barris GUB de dim_territorio →
# así la educación es JOINABLE con fact_incidentes_gub por barri. Normalizar la única
# variante de grafía ('el Poble-sec' vs 'el Poble Sec' que se mantuvo en dim_territorio).
BARRI_NORM = {'el Poble-sec': 'el Poble Sec'}
nom_join = edu_long['Territorio'].replace(BARRI_NORM)
map_barri = dim_territorio[dim_territorio['nivel_territorial'] == 'barri'].set_index('nom_barri')['id_territorio']
edu_long['id_territori'] = nom_join.map(map_barri).astype('Int64')
# Sólo aplica a barris; districte/municipi/'-' quedan sin id
edu_long.loc[edu_long['nivel_territorial'] != 'barri', 'id_territori'] = pd.NA

edu_long['indicador'] = 'nivell_educatiu'
edu_long['categoria'] = edu_long['Titulación académica']
edu_long['sexe'] = pd.NA
edu_long['unitat'] = 'persones'
socio_edu = edu_long[ESQ_SOCIO].copy()

n_barri = int((socio_edu['nivel_territorial'] == 'barri').sum())
n_barri_id = int(socio_edu[socio_edu['nivel_territorial'] == 'barri']['id_territori'].notna().sum())
print('socio_edu:', len(socio_edu), 'filas | niveles:', socio_edu['nivel_territorial'].value_counts(dropna=False).to_dict())
print(f'  filas barri con id_territori (GUB): {n_barri_id}/{n_barri}')
print('  titulacions:', socio_edu['categoria'].nunique())
socio_edu.head(3)

---
## 4. Unir y guardar `contexto_socioeconomico.csv`

In [ ]:
# --- Tasa de paro de CATALUÑA (INE por CCAA) — rellena el gap: el paro anterior era de
# Barcelona municipi (personas absolutas); esto es la TASA (%) de Cataluña, joinable a
# fact_criminalidad (Cataluña). Fichero latin-1, decimal coma, '09 Cataluña'.
fpar = list((S).glob('../ine/*Tasa de paro por comunidad*'))
ine_par = pd.read_csv(fpar[0], sep=';', encoding='latin-1', decimal=',')
col_t = ine_par.columns[0]
ine_par = ine_par[ine_par[col_t].astype(str).str.contains('Catalu', na=False)].copy()
socio_paro_cat = pd.DataFrame({
    'anyo': ine_par['Periodo'].astype(int),
    'id_territori': pd.array([ID_CATALUNA] * len(ine_par), dtype='Int64'),
    'territori': 'Cataluña',
    'nivel_territorial': 'ccaa',
    'indicador': 'taxa_atur',
    'categoria': pd.NA,
    'sexe': pd.NA,
    'valor': pd.to_numeric(ine_par['Total'], errors='coerce'),
    'unitat': 'percentatge'
})[ESQ_SOCIO]
print('socio_paro_cat (tasa paro Cataluña):', len(socio_paro_cat), 'filas | años', socio_paro_cat['anyo'].min(), '-', socio_paro_cat['anyo'].max())

contexto_socio = pd.concat([socio_renta, socio_paro, socio_paro_cat, socio_edu], ignore_index=True)
contexto_socio['anyo'] = contexto_socio['anyo'].astype(int)
contexto_socio['id_territori'] = contexto_socio['id_territori'].astype('Int64')

print('\ncontexto_socioeconomico:', contexto_socio.shape)
print('Filas por indicador:')
print(contexto_socio['indicador'].value_counts())
print('Nulos valor:', contexto_socio['valor'].isna().sum())
fk = contexto_socio['id_territori'].dropna()
print('id_territori (no nulo) válido en dim_territorio:', fk.isin(dim_territorio['id_territorio']).all(), f'| filas con id: {len(fk)}')
print('Niveles territoriales:', contexto_socio['nivel_territorial'].value_counts(dropna=False).to_dict())

contexto_socio.to_csv(CLEAN / 'contexto_socioeconomico.csv', index=False, encoding='utf-8')
print('\n[OK] Guardado contexto_socioeconomico.csv')

---
## 5. Población → `contexto_poblacion.csv` (Padrón Dataset 1)

El padrón ya trae la fila agregada **'Catalunya'** (no hay que sumar municipios).
`estado='..'` marca valores no disponibles → se eliminan.

In [ ]:
pad = pd.read_csv(S / 'Dataset 1 Padrón municipal de habitantes.csv', sep=';', encoding='utf-8-sig', low_memory=False)
pad = pad.rename(columns={'año': 'anyo', 'municipio': 'territori', 'sexo': 'sexe_raw'})
# Eliminar valores no disponibles ('..' / NaN)
pad = pad[pad['valor'].notna()].copy()
pad['valor'] = pad['valor'].astype(int)
pad['sexe'] = pad['sexe_raw'].map(SEXE_MAP)
# Nivel territorial: 'Catalunya' es el agregado CCAA; el resto son municipios
pad['nivel_territorial'] = np.where(pad['territori'] == 'Catalunya', 'ccaa', 'municipi')
pad['territori'] = pad['territori'].replace({'Catalunya': 'Cataluña'})
pad['id_territori'] = np.where(pad['nivel_territorial'] == 'ccaa', ID_CATALUNA, pd.NA)
pad['id_territori'] = pad['id_territori'].astype('Int64')
contexto_pob = pad[ESQ_POB].copy()

# Validación: por (anyo, territori) → Homes + Dones == Total
piv = contexto_pob.pivot_table(index=['anyo','territori'], columns='sexe', values='valor', aggfunc='sum')
piv = piv.dropna(subset=['Homes','Dones','Total'])
n_mal = ((piv['Homes'] + piv['Dones']) != piv['Total']).sum()
print('contexto_poblacion:', contexto_pob.shape)
print('  años:', contexto_pob['anyo'].min(), '-', contexto_pob['anyo'].max(), '| territoris:', contexto_pob['territori'].nunique())
print('  (anyo,territori) donde Homes+Dones != Total:', n_mal)
print('  sexes:', sorted(contexto_pob['sexe'].dropna().unique()))
# Cross-check: población Cataluña total 2023
cat23 = contexto_pob[(contexto_pob['territori']=='Cataluña') & (contexto_pob['anyo']==2023) & (contexto_pob['sexe']=='Total')]['valor']
print('  Población Cataluña 2023 (Total):', cat23.values)

In [ ]:
contexto_pob.to_csv(CLEAN / 'contexto_poblacion.csv', index=False, encoding='utf-8')
print(f'[OK] Guardado contexto_poblacion.csv: {len(contexto_pob)} filas')
print('\nListo para continuar con 07_etl_encuestas.ipynb')